# 01.9 - Union-Find (Disjoint Set Union, DSU)

## What is it?
A structure that keeps track of **groups** (disjoint sets) and supports two operations very fast:
- `find(x)` → which group is `x` in? (returns the group's representative / root)
- `union(x, y)` → merge the groups of `x` and `y`

Each group is a tree; each element points to its **parent**; the root represents the group.

```
union(1,2), union(3,4), union(2,4):

  before:  1  2  3  4      after:     1
                                     / \
                                    2   3
                                        |
                                        4
```

Two optimisations make it almost O(1) per operation (amortised O(α(n)), α ≤ 4 for any practical n):
1. **Path compression** - on `find`, point nodes directly at the root.
2. **Union by rank/size** - attach the smaller tree under the bigger one.

## Where is it used?
| Real system | Use |
|---|---|
| **Network connectivity** | Are two computers connected? Merge networks |
| **Kruskal's Minimum Spanning Tree** (see 02.4) | Cheapest way to connect cities with cables/roads/pipes |
| **Social networks** | Friend circles / communities |
| **Image processing** | Connected regions of pixels, segmentation |
| **Account merging** | Same person with multiple emails (LC #721) |
| **Compilers** | Type inference (unification) |
| **Percolation / physics simulations** | Do clusters connect top to bottom? |

## 1. Build it from scratch

In [1]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))   # each node is its own root
        self.size = [1] * n
        self.groups = n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path compression (halving)
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                                   # already in the same group
        if self.size[ra] < self.size[rb]:                  # union by size
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        self.groups -= 1
        return True

    def connected(self, a, b):
        return self.find(a) == self.find(b)

uf = UnionFind(6)
uf.union(0, 1); uf.union(2, 3); uf.union(1, 3)
print("0 and 3 connected?", uf.connected(0, 3))
print("0 and 5 connected?", uf.connected(0, 5))
print("number of groups:", uf.groups)      # {0,1,2,3}, {4}, {5}
assert uf.groups == 3

0 and 3 connected? True
0 and 5 connected? False
number of groups: 3


---
## Solved problems

### 1. Number of Provinces - LC #547 (Medium)
`isConnected[i][j] = 1` if cities i and j are directly connected. Count connected groups.

**Idea:** Union every connected pair; the number of remaining groups is the answer. (DFS also works.)

In [2]:
def find_circle_num(is_connected):
    n = len(is_connected)
    uf = UnionFind(n)
    for i in range(n):
        for j in range(i + 1, n):
            if is_connected[i][j]:
                uf.union(i, j)
    return uf.groups

assert find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2
assert find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
print("Number of Provinces ")

Number of Provinces 


### 2. Redundant Connection - LC #684 (Medium) A tree had one extra edge added. Return the edge that creates a cycle.

**Idea:** Add edges one by one; if both ends are already in the same group, this edge closes a cycle.

In [3]:
def find_redundant_connection(edges):
    uf = UnionFind(len(edges) + 1)
    for u, v in edges:
        if not uf.union(u, v):          # already connected -> this edge makes a cycle
            return [u, v]

assert find_redundant_connection([[1, 2], [1, 3], [2, 3]]) == [2, 3]
assert find_redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]) == [1, 4]
print("Redundant Connection ")

Redundant Connection 


### 3. Accounts Merge - LC #721 (Medium) - real-world identity resolution
Each account = [name, emails...]. Accounts sharing any email belong to the same person. Merge them.

**Idea:** Union accounts that share an email (map email → first account index), then group emails by root.

In [4]:
from collections import defaultdict

def accounts_merge(accounts):
    uf = UnionFind(len(accounts))
    email_owner = {}
    for i, (_, *emails) in enumerate(accounts):
        for email in emails:
            if email in email_owner:
                uf.union(i, email_owner[email])
            else:
                email_owner[email] = i
    groups = defaultdict(set)
    for email, i in email_owner.items():
        groups[uf.find(i)].add(email)
    return sorted([[accounts[root][0]] + sorted(emails) for root, emails in groups.items()])

accounts = [["John", "johnsmith@mail.com", "john_newyork@mail.com"],
            ["John", "johnsmith@mail.com", "john00@mail.com"],
            ["Mary", "mary@mail.com"],
            ["John", "johnnybravo@mail.com"]]
assert accounts_merge(accounts) == [
    ["John", "john00@mail.com", "john_newyork@mail.com", "johnsmith@mail.com"],
    ["John", "johnnybravo@mail.com"],
    ["Mary", "mary@mail.com"]]
print("Accounts Merge ")

Accounts Merge 


### 4. Number of Connected Components - LC #323 (Medium)
`n` nodes and undirected `edges`; count components.

In [5]:
def count_components(n, edges):
    uf = UnionFind(n)
    for u, v in edges:
        uf.union(u, v)
    return uf.groups

assert count_components(5, [[0, 1], [1, 2], [3, 4]]) == 2
assert count_components(5, [[0, 1], [1, 2], [2, 3], [3, 4]]) == 1
print("Connected Components ")

Connected Components 


## More practice on LeetCode
Medium: Graph Valid Tree (#261), Most Stones Removed (#947), Satisfiability of Equality Equations (#990), Smallest String With Swaps (#1202)
Hard: Number of Islands II (#305), Swim in Rising Water (#778)

## Key takeaways
- Use Union-Find when the question is about **groups / connectivity that keeps growing** (edges added over time).
- `union` returning `False` = the two were already connected = **cycle detected**.
- With path compression + union by size, operations are effectively O(1).

**Next: `02-Algorithms/02.1-Recursion.ipynb`**